# Тестирование сервиса в ручную

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 500) 

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()
load_dotenv(env_path)

HF_TOKEN = os.getenv("HF_TOKEN")

print(env_path)
print(HF_TOKEN is not None)

D:\Projects\speech_to_text_service\.env
True


In [3]:
import sys
import os

project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)

from app.models.transcriber import FastWhisperTranscriber
from app.settings import settings

# TEST_AUDIO_FILE = settings.AUDIO_DIR / "test_video_2.webm"

TEST_AUDIO_FILE = settings.AUDIO_DIR / "test_video_3.mp4"


# MODEL_NAME = "bzikst/faster-whisper-large-v3-russian"
# MODEL_NAME = "bzikst/faster-whisper-large-v3-russian-int8"

# MODEL_NAME = "Systran/faster-whisper-tiny"
# MODEL_NAME = "Systran/faster-whisper-small"
# MODEL_NAME = "Systran/faster-whisper-medium"
# MODEL_NAME = "Systran/faster-whisper-large-v3"

MODEL_NAME = "large-v3"

# MODEL_NAME = "dvislobokov/faster-whisper-large-v3-turbo-russian"

In [4]:
from app.service.source_downloader import download_source

source_url = "https://www.youtube.com/watch?v=YupO_OH10aI"


TEST_AUDIO_FILE = settings.AUDIO_DIR / "test_video_5.mp4"

if TEST_AUDIO_FILE.exists() and TEST_AUDIO_FILE.stat().st_size > 0:
    print("Файл уже существует, скачивание пропущено:", TEST_AUDIO_FILE)
else:
    downloaded_path = download_source(source_url)

    if downloaded_path != TEST_AUDIO_FILE:
        TEST_AUDIO_FILE.write_bytes(downloaded_path.read_bytes())
        downloaded_path.unlink(missing_ok=True)
    else:
        TEST_AUDIO_FILE = downloaded_path

    print("Файл скачан:", TEST_AUDIO_FILE)

print("Размер, МБ:", TEST_AUDIO_FILE.stat().st_size / 1024 / 1024)

Файл уже существует, скачивание пропущено: D:\Projects\speech_to_text_service\data\audio\test_video_5.mp4
Размер, МБ: 23.46971893310547


In [5]:
from pydub import AudioSegment

def get_audio_duration(path: str) -> float:
    audio = AudioSegment.from_file(path)
    return len(audio) / 1000.0

print(f"Audio duration: {get_audio_duration(str(TEST_AUDIO_FILE))} seconds")

Audio duration: 572.883 seconds


In [6]:
transcriber = FastWhisperTranscriber(
    model_name=MODEL_NAME,
    cache_dir=settings.CACHE_DIR,
    # device="cpu",
    device="cuda",
    # compute_type="int8",
    # cpu_threads=4,
    # num_workers=8, 
    token=HF_TOKEN,   
)


2026-07-16 15:53:28 - model.large-v3 - INFO - MainProcess[10516] - Модель загружена: large-v3


In [7]:
result = transcriber.transcribe(
    audio_path=str(TEST_AUDIO_FILE),
    # language="ru",
    task="transcribe",

    beam_size=3,
    best_of=3,
    # temperature=0.0,

    vad_filter=True,
    vad_parameters={
        "min_silence_duration_ms": 500,
        "speech_pad_ms": 200,
    },

    condition_on_previous_text=False,

    initial_prompt=(
        "Ставь точки, запятые, вопросительные знаки и дели текст на предложения."
    ),

    word_timestamps=False,
    without_timestamps=False,
)

2026-07-16 15:53:28 - model.large-v3 - INFO - MainProcess[10516] - Транскрибация начата | модель=large-v3 | файл=D:\Projects\speech_to_text_service\data\audio\test_video_5.mp4 | язык=None | задача=transcribe
2026-07-16 15:53:30 - faster_whisper - INFO - MainProcess[10516] - Processing audio with duration 09:32.883
2026-07-16 15:53:32 - faster_whisper - INFO - MainProcess[10516] - VAD filter removed 02:57.235 of audio
2026-07-16 15:53:35 - faster_whisper - INFO - MainProcess[10516] - Detected language 'ru' with probability 1.00
2026-07-16 15:53:35 - model.large-v3 - INFO - MainProcess[10516] - Длительность аудио: 572.88с.
2026-07-16 15:53:40 - model.large-v3 - INFO - MainProcess[10516] - Прогресс | модель=large-v3 | файл=test_video_5.mp4 | выполнено=5.3%
2026-07-16 15:53:40 - model.large-v3 - INFO - MainProcess[10516] - Прогресс | модель=large-v3 | файл=test_video_5.mp4 | выполнено=9.2%
2026-07-16 15:53:45 - model.large-v3 - INFO - MainProcess[10516] - Прогресс | модель=large-v3 | файл=

In [8]:
result

{'language': 'ru',
 'duration': 572.88275,
 'text': 'Ольга Санна, слушайте меня. Когда мы его поймаем, ни о какой Женевской конвенции я не хочу слышать. Просто отдадите его мне на растерзание. У вас что, арбалет? Да, еще светошумовая граната, сюрикены и песок, что бросить ему в глаза. Денис, пусть сюрикены, вы что, мы датчика ищем или Рэмбо? Задолбали эти закладчики. Да посмотрите, как они нам двор весь перерыли, у нас двор весь в ямках, как лицо студента. Ну все, я обход закончил, вроде все спокойно. Спокойно? А что так запыхались? Да там собаки слиплись, я разлеплял. Блин, поймали бы уже этого закладчика, надоел он. Постоянно наркоманы эти приходят к нам во двор, колют тут свои марихуаны, пьют кокаины свои, надоели уже. Во дворе столько наркоты примагничено, что магнитное поле образовалось. Вон счетчики все в квартире повылетали. Согласен, с алкашами было проще. Они заметные, они шумные. И у них у всех на рубашке вот такой карман есть, чтобы платок класть или сигареты. А вот наркоман

In [9]:
from app.models.sherpa_speaker_diarization import SherpaOnnxSpeakerDiarizationModel
from app.utils.exporters.common import build_speaker_blocks, build_paragraph_blocks
from app.settings import settings

diarizer = SherpaOnnxSpeakerDiarizationModel(
    cache_dir=settings.CACHE_DIR,
    token=settings.HF_TOKEN,
    provider="cpu",
    num_threads=4,
    cluster_threshold=1.0,
    min_duration_on=0.1,
    min_duration_off=0.1,

)

diarization_result = diarizer.diarize(
    audio_path=str(TEST_AUDIO_FILE),
    num_speakers=None,
)

result["diarization"] = diarization_result
result["diarization_error"] = None

print("=== DIARIZATION ===")
print("duration:", diarization_result["duration"])
print("num_speakers:", diarization_result["num_speakers"])
print("speakers:", diarization_result["speakers"])
print("segments:", len(diarization_result["segments"]))

print("\n=== FIRST DIARIZATION SEGMENTS ===")
for segment in diarization_result["segments"]:
    print(
        f'{segment["speaker"]}: '
        f'{segment["start"]:.2f} -> {segment["end"]:.2f} '
        f'({segment["duration"]:.2f}s)'
    )



2026-07-16 15:54:54 - diarization.sherpa_onnx - INFO - MainProcess[10516] - Модели Sherpa-ONNX diarization готовы | segmentation=D:\Projects\speech_to_text_service\data\cache_dir\models--csukuangfj--sherpa-onnx-pyannote-segmentation-3-0\snapshots\9403a6902bb58e3d5ae8c7e77c3422de279db2e0\model.onnx | embedding=D:\Projects\speech_to_text_service\data\cache_dir\models--csukuangfj--speaker-embedding-models\snapshots\0743f301363dec56491a490f6d6cbc9d67f9a3bf\nemo_en_titanet_small.onnx
2026-07-16 15:54:55 - diarization.sherpa_onnx - INFO - MainProcess[10516] - Sherpa-ONNX diarizer инициализирован | num_clusters=-1 | sample_rate=16000 | provider=cpu | num_threads=4
2026-07-16 15:54:56 - diarization.sherpa_onnx - INFO - MainProcess[10516] - Sherpa-ONNX diarization запущен | файл=D:\Projects\speech_to_text_service\data\audio\test_video_5.mp4 | num_speakers=None | sample_rate=16000 | provider=cpu
2026-07-16 15:56:03 - diarization.sherpa_onnx - INFO - MainProcess[10516] - Sherpa-ONNX diarization з

In [10]:
import json
from pathlib import Path

RESULT_JSON_FILE = settings.AUDIO_DIR / "debug_transcription_with_diarization.json"

with RESULT_JSON_FILE.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Result saved:", RESULT_JSON_FILE)

Result saved: D:\Projects\speech_to_text_service\data\audio\debug_transcription_with_diarization.json


In [11]:
import json

with RESULT_JSON_FILE.open("r", encoding="utf-8") as f:
    result = json.load(f)

print("Result loaded:", RESULT_JSON_FILE)
print("segments:", len(result.get("segments") or []))
print("diarization segments:", len((result.get("diarization") or {}).get("segments") or []))

Result loaded: D:\Projects\speech_to_text_service\data\audio\debug_transcription_with_diarization.json
segments: 140
diarization segments: 155


In [12]:
from app.utils.exporters.common import build_speaker_blocks

speaker_blocks = build_speaker_blocks(result)

print("\n=== TEXT + SPEAKERS ===")
for block in speaker_blocks:
    speaker = block.get("speaker") or "Без спикера"
    parts = block.get("parts") or []
    if not parts:
        continue

    start = parts[0].get("start")
    end = parts[-1].get("end")
    text = " ".join(part["text"] for part in parts)

    print(f"\n{speaker} [{start:.2f} - {end:.2f}]")
    print(text)


=== TEXT + SPEAKERS ===

Спикер 01 [19.03 - 30.10]
Ольга Санна, слушайте меня. Когда мы его поймаем, ни о какой Женевской конвенции я не хочу слышать. Просто отдадите его мне на растерзание.

Спикер 02 [31.06 - 31.96]
У вас что, арбалет?

Спикер 01 [32.69 - 36.59]
Да, еще светошумовая граната, сюрикены и песок, что бросить ему в глаза.

Спикер 02 [37.80 - 52.95]
Денис, пусть сюрикены, вы что, мы датчика ищем или Рэмбо? Задолбали эти закладчики. Да посмотрите, как они нам двор весь перерыли, у нас двор весь в ямках, как лицо студента.

Спикер 03 [52.95 - 89.33]
Ну все, я обход закончил, вроде все спокойно. Спокойно? А что так запыхались? Да там собаки слиплись, я разлеплял. Блин, поймали бы уже этого закладчика, надоел он. Постоянно наркоманы эти приходят к нам во двор, колют тут свои марихуаны, пьют кокаины свои, надоели уже. Во дворе столько наркоты примагничено, что магнитное поле образовалось.

Спикер 01 [89.33 - 121.35]
Вон счетчики все в квартире повылетали. Согласен, с алкашами 